# Oracle Knowledge RAG Ingestion

Standalone notebook for loading Oracle Knowledge articles from the `ID` column in `KM_updated.xlsx`, extracting the rendered article content with Playwright, chunking it, and optionally building a Chroma index.

This notebook is independent of the slower Oracle documentation crawler in `RAG_v1.ipynb`.

In [ ]:
# CELL 1: Install dependencies (run once per Jupyter environment)
%pip install -q openpyxl playwright beautifulsoup4 lxml langchain-core langchain-text-splitters
!playwright install chromium

In [ ]:
# CELL 2: Configuration
from pathlib import Path

KM_EXCEL_FILE = Path("KM_updated.xlsx") if Path("KM_updated.xlsx").exists() else Path("KM.xlsx")
KM_ID_COLUMN = "ID"
KM_TITLE_COLUMN = "Title"
KM_URL_TEMPLATE = (
    "https://fa-etmi-saasfaprod1.fa.ocs.oraclecloud.com/"
    "fscmUI/redwood/service/ec/container/sr/view-article?docId={}"
)

# First test: use 5 or 10. Set to None later for all IDs.
KM_MAX_ARTICLES = 10

# Persistent browser profile keeps Oracle SSO cookies between runs.
KM_AUTH_PROFILE = Path("./oracle_km_browser_profile")

# Oracle pages can render client-side. Increase if articles load slowly.
KM_WAIT_AFTER_LOAD = 5
PAGE_TIMEOUT_MS = 60000

# Chunking
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Output files
RAW_JSONL = Path("oracle_km_articles.jsonl")
CHUNKS_JSONL = Path("oracle_km_chunks.jsonl")
CHROMA_DIR = Path("./vectordb_oracle_knowledge")

print("Workbook:", KM_EXCEL_FILE.resolve())
print("Max articles:", KM_MAX_ARTICLES)

In [ ]:
# CELL 3: Imports and ID loading
import json
import re
import os
from pathlib import Path

from openpyxl import load_workbook
from bs4 import BeautifulSoup
from langchain_core.documents import Document


def read_km_articles(excel_file=KM_EXCEL_FILE,
                     id_column=KM_ID_COLUMN,
                     title_column=KM_TITLE_COLUMN):
    wb = load_workbook(excel_file, read_only=True, data_only=True)

    articles = []
    seen = set()

    try:
        for ws in wb.worksheets:
            headers = {
                str(cell.value).strip(): cell.column
                for cell in ws[1]
                if cell.value is not None
            }

            if id_column not in headers:
                continue

            id_col = headers[id_column]
            title_col = headers.get(title_column)

            for row in range(2, ws.max_row + 1):
                value = ws.cell(row=row, column=id_col).value
                if value is None:
                    continue

                doc_id = str(value).strip()
                if not doc_id or doc_id in seen:
                    continue

                title = ""
                if title_col:
                    title_value = ws.cell(
                        row=row,
                        column=title_col
                    ).value
                    if title_value:
                        title = str(title_value).strip()

                seen.add(doc_id)
                articles.append({
                    "doc_id": doc_id,
                    "title": title,
                })
    finally:
        wb.close()

    return articles


KM_ARTICLES = read_km_articles()
print(f"Oracle Knowledge IDs found: {len(KM_ARTICLES)}")
print("First 10:", KM_ARTICLES[:10])

## Oracle authentication

Run the next cell once. A Chromium window will open. Complete Oracle SSO/login there and make sure the first Knowledge article is visible. Then return to Jupyter and continue with the extraction cell. The browser session is persisted in `oracle_km_browser_profile/`.

In [ ]:
# CELL 4: Open a persistent Oracle browser for SSO
from playwright.async_api import async_playwright

playwright = await async_playwright().start()
browser_context = await playwright.chromium.launch_persistent_context(
    user_data_dir=str(KM_AUTH_PROFILE),
    headless=False,
    viewport={"width": 1440, "height": 1000},
    accept_downloads=False,
)

km_page = await browser_context.new_page()
km_page.set_default_timeout(PAGE_TIMEOUT_MS)

first_id = DOC_IDS[0]
first_url = KM_URL_TEMPLATE.format(first_id)
await km_page.goto(first_url, wait_until="domcontentloaded", timeout=PAGE_TIMEOUT_MS)
await km_page.wait_for_timeout(KM_WAIT_AFTER_LOAD * 1000)

print("Browser opened:", km_page.url)
print("Complete Oracle SSO/login in the browser window if needed.")
print("When the article is visible, move to CELL 5. No terminal/input prompt is required.")

In [ ]:
# CELL 5: Check authentication manually
print("Current URL:", km_page.url)
print("Page title:", await km_page.title())
print("
If you see the Oracle Knowledge article in the browser, continue to CELL 6.")

In [ ]:
# CELL 6: Extract article text one-by-one
async def clean_html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "noscript", "svg", "nav", "footer"]):
        tag.decompose()
    text = soup.get_text("\n", strip=True)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


async def extract_current_article(page, doc_id, excel_title, url):
    title = excel_title or ""

    selectors = [
        "article",
        "main",
        '[role="main"]',
        '[data-testid*="article" i]',
        '[class*="article" i]',
        '[class*="knowledge" i]',
    ]

    best_html = ""
    best_len = 0
    for selector in selectors:
        try:
            loc = page.locator(selector)
            count = await loc.count()
            for i in range(min(count, 8)):
                html = await loc.nth(i).inner_html()
                text_len = len(await clean_html_to_text(html))
                if text_len > best_len:
                    best_html = html
                    best_len = text_len
        except Exception:
            continue

    if not best_html:
        best_html = await page.locator("body").inner_html()

    text = await clean_html_to_text(best_html)
    if title and title not in text:
        text = f"{title}\n\n{text}"

    return {
        "doc_id": doc_id,
        "title": title,
        "url": url,
        "text": text,
        "content_type": "oracle_knowledge_article",
    }


articles_to_process = KM_ARTICLES[:KM_MAX_ARTICLES] if KM_MAX_ARTICLES is not None else KM_ARTICLES
print(f"Processing {len(articles_to_process)} articles...")

articles = []
failed = []

for i, article_info in enumerate(articles_to_process, 1):
    doc_id = article_info["doc_id"]
    excel_title = article_info["title"]
    url = KM_URL_TEMPLATE.format(doc_id)
    print(f"[{i}/{len(articles_to_process)}] {doc_id} - {excel_title}")
    try:
        await km_page.goto(url, wait_until="domcontentloaded", timeout=PAGE_TIMEOUT_MS)
        await km_page.wait_for_timeout(KM_WAIT_AFTER_LOAD * 1000)

        # Give client-side rendering a chance to finish.
        try:
            await km_page.wait_for_load_state("networkidle", timeout=15000)
        except Exception:
            pass

        current_url = km_page.url.lower()
        if "login" in current_url or "signin" in current_url:
            raise RuntimeError("Oracle session appears to have expired or redirected to login")

        article = await extract_current_article(km_page, doc_id, excel_title, url)

        if len(article["text"]) < 100:
            raise RuntimeError(f"Extracted content is unusually short ({len(article['text'])} chars)")

        articles.append(article)
        print(f"    extracted {len(article['text']):,} characters")

    except Exception as exc:
        failed.append({"doc_id": doc_id, "url": url, "error": str(exc)})
        print(f"    FAILED: {exc}")

print(f"\nSuccessful: {len(articles)}")
print(f"Failed: {len(failed)}")

In [ ]:
# CELL 6A: Validate titles from KM_updated.xlsx
print("Sample extracted metadata:")
for article in articles[:10]:
    print(
        "ID:", article["doc_id"],
        "| Title:", article["title"],
        "| URL:", article["url"],
    )

assert all(a["title"] for a in articles), (
    "At least one article has an empty Excel Title."
)


In [ ]:
# CELL 7: Save raw extracted articles
with RAW_JSONL.open("w", encoding="utf-8") as f:
    for article in articles:
        f.write(json.dumps(article, ensure_ascii=False) + "\n")

with open("oracle_km_failed.json", "w", encoding="utf-8") as f:
    json.dump(failed, f, ensure_ascii=False, indent=2)

print("Saved:", RAW_JSONL.resolve())
print("Failed log:", Path("oracle_km_failed.json").resolve())

In [ ]:
# CELL 8: Inspect an extracted article BEFORE chunking
if articles:
    sample = articles[0]
    print("DOC ID:", sample["doc_id"])
    print("TITLE:", sample["title"])
    print("URL:", sample["url"])
    print("\nTEXT PREVIEW:\n")
    print(sample["text"][:3000])
else:
    print("No successful articles were extracted.")

In [ ]:
# CELL 9: Convert to LangChain Documents and chunk
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    Document(
        page_content=a["text"],
        metadata={
            "doc_id": a["doc_id"],
            "title": a["title"],
            "source": a["url"],
            "platform": "Oracle Knowledge",
            "content_type": a["content_type"],
        },
    )
    for a in articles
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

chunks = text_splitter.split_documents(documents)
print(f"Source documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")
if chunks:
    print("Sample metadata:", chunks[0].metadata)
    print("Sample chunk:\n", chunks[0].page_content[:1500])

In [ ]:
# CELL 10: Save chunks as JSONL for indexing or inspection
with CHUNKS_JSONL.open("w", encoding="utf-8") as f:
    for idx, chunk in enumerate(chunks):
        record = {
            "chunk_id": f"{chunk.metadata['doc_id']}_{idx}",
            "doc_id": chunk.metadata["doc_id"],
            "title": chunk.metadata["title"],
            "source": chunk.metadata["source"],
            "platform": chunk.metadata["platform"],
            "text": chunk.page_content,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved:", CHUNKS_JSONL.resolve())

In [ ]:
# CELL 11: OPTIONAL - build Chroma index
# Run this only after you verify CELL 8 output looks like clean article content.
%pip install -q langchain-huggingface langchain-chroma sentence-transformers

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

if CHROMA_DIR.exists():
    import shutil
    shutil.rmtree(CHROMA_DIR)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=str(CHROMA_DIR),
    collection_name="oracle_knowledge",
)

print("Chroma index created:", CHROMA_DIR.resolve())
print("Chunk count:", vector_db._collection.count())

In [ ]:
# CELL 12: OPTIONAL - close the browser when finished
await browser_context.close()
await playwright.stop()
print("Oracle browser session closed.")